# Nano-NLA Stage 2 CPU/API windows

Use this notebook on a CPU Colab runtime after Stage 0 output has been copied into the Drive-backed `data/generated` directory. Each Stage 2 increment asks hosted summary providers for up to 20k new AV-SFT input rows and up to 20k new AR-SFT input rows. Re-run the same Stage 2 cell after a Colab or provider interruption; prompt-fingerprinted chunk checkpoints stay on Drive.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/nano-nla')
REPO_DIR = Path('/content/Nano-NLA')
REPO_URL = 'https://github.com/IrohAmca/Nano-NLA.git'  # Change this if using a fork.
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

if not (REPO_DIR / '.git').exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --ff-only
import os
os.chdir(REPO_DIR)
print('cwd:', Path.cwd())

In [ ]:
!pip -q install uv
!uv sync --extra groq

from pathlib import Path

def link_drive_dir(local: Path, persistent: Path) -> None:
    persistent.mkdir(parents=True, exist_ok=True)
    local.parent.mkdir(parents=True, exist_ok=True)
    if local.is_symlink():
        if local.resolve() == persistent.resolve():
            return
        local.unlink()
    elif local.exists():
        raise RuntimeError(f'{local} exists in the Colab checkout; move it before linking Drive artifacts.')
    local.symlink_to(persistent, target_is_directory=True)

link_drive_dir(Path('data/generated'), DRIVE_ROOT / 'data' / 'generated')
link_drive_dir(Path('checkpoints'), DRIVE_ROOT / 'checkpoints')
link_drive_dir(Path('results'), DRIVE_ROOT / 'results')
print('Drive artifacts:', DRIVE_ROOT)

In [ ]:
import getpass
import os

for key_name in ('DEEPSEEK_API_KEY', 'GROQ_API_KEY', 'NVIDIA_API_KEY'):
    if os.environ.get(key_name):
        print(f'{key_name}: already set')
        continue
    value = getpass.getpass(f'{key_name} (blank to skip): ')
    if value:
        os.environ[key_name] = value
print('The multi provider will skip hosted providers without keys.')

In [ ]:
from pathlib import Path
import pyarrow.parquet as pq

GENERATED = Path('data/generated')
BASE = GENERATED / 'base.parquet'
BASE_META = GENERATED / 'base.parquet.nla_meta.yaml'
assert BASE.exists(), f'Missing Stage 0 base parquet: {BASE}'
assert BASE_META.exists(), f'Missing Stage 0 sidecar: {BASE_META}'

def parquet_rows(path: Path) -> str:
    return str(pq.ParquetFile(path).metadata.num_rows) if path.exists() else 'missing'

for rel_path in (
    'base.parquet',
    'splits/av_sft_raw.parquet',
    'splits/ar_sft_raw.parquet',
    'splits/rl_raw.parquet',
    'splits/av_sft_explained.parquet',
    'splits/ar_sft_explained.parquet',
    'av_sft.parquet',
    'ar_sft.parquet',
    'rl.parquet',
):
    path = GENERATED / rel_path
    print(f'{rel_path:38} {parquet_rows(path)}')

Run Stage 1 only when the raw split parquets are missing. It reads the Stage 0 sidecar and writes the AV-SFT, AR-SFT, and RL raw splits under `data/generated/splits`.

In [ ]:
raw_splits = [
    GENERATED / 'splits' / 'av_sft_raw.parquet',
    GENERATED / 'splits' / 'ar_sft_raw.parquet',
    GENERATED / 'splits' / 'rl_raw.parquet',
]
if any(not path.exists() for path in raw_splits):
    !uv run python scripts/run_datagen.py --config configs/qwen05b.yaml --stage 1
else:
    print('Stage 1 raw splits already exist; leaving them unchanged.')

This is the resumable Stage 2 increment. Keep the default checkpoint directories per split. Do not point AV and AR at one shared `--summary-checkpoint-dir`.

In [ ]:
!uv run python scripts/run_datagen.py \
  --config configs/qwen05b.yaml \
  --stage 2 \
  --summary-provider multi \
  --summary-max-new-rows 20000

In [ ]:
!uv run python scripts/run_datagen.py --config configs/qwen05b.yaml --stage 3

In [ ]:
from pathlib import Path
import pyarrow.parquet as pq

def parquet_rows(path: Path) -> str:
    return str(pq.ParquetFile(path).metadata.num_rows) if path.exists() else 'missing'

for rel_path in (
    'splits/av_sft_explained.parquet',
    'splits/ar_sft_explained.parquet',
    'av_sft.parquet',
    'ar_sft.parquet',
    'rl.parquet',
):
    path = Path('data/generated') / rel_path
    print(f'{rel_path:38} {parquet_rows(path)}')